# 01 — Exploratory Data Analysis
**Model Risk Governance Toolkit | Lending Club Credit Risk**

This notebook explores the raw Lending Club dataset before any modelling. The goals are:
1. Understand class balance and default rates over time
2. Identify key feature distributions and their relationship to default
3. Detect missing value patterns
4. Understand temporal patterns (volume, FICO, DTI over time)

**SR 11-7 context:** Data exploration is the first step of the model development lifecycle.
The validator expects to see evidence that the developer understood the data before building the model.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from toolkit.data_validation import validate
from toolkit.preprocessing import engineer_features

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')
print('Libraries loaded.')

## 1. Load and Validate Data

In [ ]:
# Load raw data — low_memory=False needed because Lending Club has mixed-type columns
raw = pd.read_csv('../data/raw/lending_club.csv', low_memory=False)
print(f'Raw shape: {raw.shape}')
raw.head(3)

In [ ]:
# Parse issue date — format '%b-%Y' = e.g. 'Jan-2012'
raw['issue_d'] = pd.to_datetime(raw['issue_d'], format='%b-%Y', errors='coerce')

# Run validation: binarize target, drop leakage columns, flag missing rates
df = validate(raw.copy())
df = engineer_features(df)
print(f'\nClean shape: {df.shape}')
print(f'Date range: {df["issue_d"].min().date()} → {df["issue_d"].max().date()}')

## 2. Class Balance

> **Learning annotation:** Credit datasets are almost always imbalanced — defaults are
> typically 15–25% of originated loans. This means accuracy is a misleading metric:
> a model that predicts 'no default' for every loan achieves 80%+ accuracy while
> being completely useless. We use AUC-ROC and Gini instead.
>
> The `class_weight='balanced'` parameter in Logistic Regression compensates for
> this imbalance during training.

In [ ]:
print('=== Overall Class Balance ===')
print(df['default'].value_counts())
print(f'\nDefault rate: {df["default"].mean():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['default'].value_counts()
axes[0].pie(counts, labels=['Fully Paid', 'Charged Off'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90)
axes[0].set_title('Class Balance')

# Default rate by year
df['year'] = df['issue_d'].dt.year
yearly = df.groupby('year')['default'].mean().reset_index()
axes[1].bar(yearly['year'], yearly['default'],
            color=['#e74c3c' if y in [2008, 2009] else '#3498db' for y in yearly['year']])
axes[1].set_xlabel('Issue Year')
axes[1].set_ylabel('Default Rate')
axes[1].set_title('Default Rate by Year')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.show()

## 3. The 2008–2009 Default Spike — Concept Drift

> **Learning annotation — Concept Drift:**
> The bars above show a notable spike in default rates during 2008–2009.
> This is the U.S. financial crisis, and it is a canonical example of **concept drift**:
> the statistical relationship between borrower features (FICO, DTI, income)
> and default probability changed fundamentally during the crisis.
>
> A model trained on pre-2008 data would have dramatically underestimated default
> probabilities in 2008–2009, not because the features were wrong, but because the
> world changed. SR 11-7 requires banks to monitor models for exactly this kind
> of environmental shift and retrain or recalibrate when it occurs.
>
> In this project, our temporal split (train < 2015, monitor ≥ 2015) tests whether
> the model trained on earlier data remains valid as the loan population evolves.

## 4. Temporal Volume Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loan volume by year
vol = df.groupby('year').size().reset_index(name='count')
axes[0].bar(vol['year'], vol['count'], color='#3498db')
axes[0].set_title('Loan Volume by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Loans')

# Avg FICO by year
fico = df.groupby('year')['fico_mid'].mean()
axes[1].plot(fico.index, fico.values, 'o-', color='#9b59b6')
axes[1].set_title('Average FICO by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Avg FICO Score')

# Avg DTI by year
dti = df.groupby('year')['dti'].mean()
axes[2].plot(dti.index, dti.values, 'o-', color='#e67e22')
axes[2].set_title('Average DTI by Year')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Debt-to-Income Ratio')

plt.tight_layout()
plt.show()

## 5. Feature Distributions — Defaults vs Non-Defaults

In [ ]:
top_numeric = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_mid',
               'installment', 'revol_util', 'inq_last_6mths', 'delinq_2yrs', 'open_acc']

available = [c for c in top_numeric if c in df.columns]
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(available[:10]):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        vals = df.loc[df['default'] == label, col].dropna()
        axes[i].hist(vals, bins=40, alpha=0.5, density=True, color=color,
                     label='Repaid' if label == 0 else 'Default')
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle('Feature Distributions: Defaults vs Non-Defaults', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 6. Missing Value Heatmap

In [ ]:
from toolkit.data_validation import missing_rate_report

missing_df = missing_rate_report(df, threshold=0.10)
flagged = missing_df[missing_df['pct_missing'] > 0.05].head(30)

if not flagged.empty:
    fig, ax = plt.subplots(figsize=(8, max(4, len(flagged) * 0.35)))
    ax.barh(flagged.index, flagged['pct_missing'],
            color=['#e74c3c' if f else '#f39c12' for f in flagged['flag']])
    ax.axvline(0.10, color='red', linestyle='--', label='10% threshold')
    ax.set_xlabel('Missing Rate')
    ax.set_title('Features with > 5% Missing Values')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No features with > 5% missing values.')

## 7. Correlation Matrix

In [ ]:
corr_cols = [c for c in top_numeric if c in df.columns] + ['default']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix (Numeric Features + Default Target)')
plt.tight_layout()
plt.show()

## 8. Temporal Train / Monitor Split Visualisation

In [ ]:
cutoff = pd.Timestamp('2015-01-01')
train_df   = df[df['issue_d'] < cutoff]
monitor_df = df[df['issue_d'] >= cutoff]

print(f'Train:   {len(train_df):>8,} loans | Default rate: {train_df["default"].mean():.2%}')
print(f'Monitor: {len(monitor_df):>8,} loans | Default rate: {monitor_df["default"].mean():.2%}')

monthly = df.set_index('issue_d').resample('ME').size().rename('loans')
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(monthly.index, monthly.values,
                where=monthly.index < cutoff, alpha=0.6, color='#3498db', label='Train')
ax.fill_between(monthly.index, monthly.values,
                where=monthly.index >= cutoff, alpha=0.6, color='#e74c3c', label='Monitor')
ax.axvline(cutoff, color='black', linestyle='--', linewidth=1.2, label='2015-01-01 cutoff')
ax.set_title('Monthly Loan Volume — Train vs Monitor Split')
ax.set_xlabel('Issue Date')
ax.set_ylabel('Loan Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save processed splits for use in notebook 02
Path('../data/processed').mkdir(parents=True, exist_ok=True)
train_df.to_parquet('../data/processed/train.parquet', index=False)
monitor_df.to_parquet('../data/processed/monitor.parquet', index=False)
print('Saved train and monitor parquet files to data/processed/')